# Exerciții Titanic — pandas, sklearn și text → numere

Serie de **10 întrebări** pentru studenți, pe baza ideilor din `02.nlp_recipes.ipynb`
(encodări, Bag of Words, TF-IDF, model Scikit-learn), aplicate pe `titanic/train.csv`.

Fiecare întrebare este urmată de **rezolvare**.

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv("titanic/train.csv")
train.head(3)

---
## Întrebarea 1

Încarcă `titanic/train.csv` și identifică:
1. care coloane sunt de tip text (`object`);
2. câte valori lipsă (`NaN`) are fiecare coloană;
3. rata de supraviețuire (`Survived.mean()`).

*Hint:* `dtypes`, `isna().sum()`, similar cu inspectarea coloanelor din notebook-ul NLP.

### Rezolvare 1

In [ ]:
print("Coloane text (object):")
print(train.select_dtypes(include="object").columns.tolist())

print("\nValori lipsă:")
print(train.isna().sum().sort_values(ascending=False))

print("\nRata de supraviețuire:", round(train["Survived"].mean(), 3))

---
## Întrebarea 2

Coloana `Age` are multe valori lipsă. Completează-le cu **mediana** folosind pandas (`fillna`).
Creează și un flag `Age_missing` (1 dacă lipsea, 0 altfel), apoi verifică că nu mai există `NaN` pe `Age`.

*Hint:* în NLP nu aveam NaN pe text; aici tratarea lipsurilor e pasul înainte de encoding.

### Rezolvare 2

In [ ]:
df = train.copy()

df["Age_missing"] = df["Age"].isna().astype(int)
age_median = df["Age"].median()
df["Age"] = df["Age"].fillna(age_median)

print(f"Mediană folosită: {age_median}")
print("Age NaN după fillna:", df["Age"].isna().sum())
print("Câți pasageri aveau Age lipsă:", df["Age_missing"].sum())
df[["Age", "Age_missing"]].head()

---
## Întrebarea 3

Transformă `Sex` și `Embarked` în coloane One-Hot cu `pd.get_dummies` (ca `mealType` în notebook-ul NLP).
Înainte, completează `Embarked` lipsă cu valoarea cea mai frecventă (`mode`).
Afișează primele 5 rânduri cu noile coloane.

### Rezolvare 3

In [ ]:
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

sex_ohe = pd.get_dummies(df["Sex"], prefix="Sex")
emb_ohe = pd.get_dummies(df["Embarked"], prefix="Emb")

ohe_preview = pd.concat([df[["Sex", "Embarked"]], sex_ohe, emb_ohe], axis=1)
ohe_preview.head()

---
## Întrebarea 4

Folosind **sklearn**:
1. `LabelEncoder` pe `Sex` → coloana `Sex_label`;
2. `OrdinalEncoder` pe `Pclass` cu ordinea `[1, 2, 3]` → coloana `Pclass_ord`.

Arată un tabel cu valorile unice (`drop_duplicates`) pentru ambele mapări.

*Hint:* la rețete, `difficulty` era ordinal (Easy < Medium); aici `Pclass` 1 e „mai bună” decât 3.

### Rezolvare 4

In [ ]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

le = LabelEncoder()
df["Sex_label"] = le.fit_transform(df["Sex"])

ord_enc = OrdinalEncoder(categories=[[1, 2, 3]])
df["Pclass_ord"] = ord_enc.fit_transform(df[["Pclass"]])

print("Sex → LabelEncoder:")
print(df[["Sex", "Sex_label"]].drop_duplicates().sort_values("Sex_label"))

print("\nPclass → OrdinalEncoder:")
print(df[["Pclass", "Pclass_ord"]].drop_duplicates().sort_values("Pclass"))

---
## Întrebarea 5

Aplică **Frequency Encoding** pe `Embarked`: fiecare port e înlocuit cu frecvența sa relativă în dataset
(`value_counts(normalize=True)` + `map`), ca în notebook-ul NLP.
Afișează frecvențele unice.

### Rezolvare 5

In [ ]:
freq = df["Embarked"].value_counts(normalize=True)
df["Embarked_freq"] = df["Embarked"].map(freq)

df[["Embarked", "Embarked_freq"]].drop_duplicates().sort_values(
    "Embarked_freq", ascending=False
)

---
## Întrebarea 6

Din coloana text `Name`, extrage **titlul** (Mr, Mrs, Miss, Master…) cu un regex pandas.
Grupează titlurile rare (&lt; 10 apariții) în categoria `"Rare"`.
Apoi One-Hot pe `Title` cu `get_dummies`.

*Hint:* pattern util: `r",\s*([^.]+)\."` — similar cu extragerea de semnal din `name` la rețete.

### Rezolvare 6

In [ ]:
df["Title"] = df["Name"].str.extract(r",\s*([^.]+)\.", expand=False)

rare = df["Title"].value_counts()
rare_titles = rare[rare < 10].index
df["Title"] = df["Title"].replace(rare_titles, "Rare")

print("Titluri după grupare:")
print(df["Title"].value_counts())

title_ohe = pd.get_dummies(df["Title"], prefix="Title")
pd.concat([df[["Name", "Title"]], title_ohe], axis=1).head(3)

---
## Întrebarea 7

Creează **feature-uri simple din text** (ca `name_len`, `name_words` în NLP) pentru:
- `Name`: lungime în caractere și număr de cuvinte;
- `Ticket`: lungime;
- `Cabin`: flag `HasCabin` (1 dacă nu e NaN) și litera punții `Deck` (prima literă; `"Unknown"` dacă lipsește).

Afișează un preview cu aceste coloane.

### Rezolvare 7

In [ ]:
df["Name_len"] = df["Name"].str.len()
df["Name_words"] = df["Name"].str.split().str.len()
df["Ticket_len"] = df["Ticket"].astype(str).str.len()

df["HasCabin"] = df["Cabin"].notna().astype(int)
df["Deck"] = df["Cabin"].str[0].fillna("Unknown")

df[["Name", "Name_len", "Name_words", "Ticket_len", "HasCabin", "Deck"]].head()

---
## Întrebarea 8

Aplică pe coloana `Name`:
1. `CountVectorizer` (Bag of Words) cu `max_features=15`, `stop_words="english"`;
2. `TfidfVectorizer` cu `max_features=15`, `stop_words="english"`.

Afișează vocabularul BoW și un DataFrame cu vectorii TF-IDF pentru primele 3 nume.

*Hint:* exact ca la `name` / `ingredients` în `02.nlp_recipes.ipynb`.

### Rezolvare 8

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count_vec = CountVectorizer(stop_words="english", max_features=15)
name_bow = count_vec.fit_transform(df["Name"])

print("Vocabular Bag of Words (Name):")
print(list(count_vec.get_feature_names_out()))

tfidf = TfidfVectorizer(stop_words="english", max_features=15)
name_tfidf = tfidf.fit_transform(df["Name"])

name_tfidf_df = pd.DataFrame(
    name_tfidf.toarray(),
    columns=[f"tfidf_{w}" for w in tfidf.get_feature_names_out()],
    index=df.index,
)
pd.concat([df[["Name"]], name_tfidf_df], axis=1).head(3)

---
## Întrebarea 9

Construiește o matrice de features `X` combinând:
- numerice: `Age`, `Fare`, `SibSp`, `Parch`, `Name_words`, `HasCabin`, `Age_missing`;
- One-Hot: `Sex`, `Embarked`, `Title` (deja calculate sau recalculate);
- TF-IDF pe `Name` (`max_features=20`).

Folosește `scipy.sparse.hstack` + `StandardScaler` pe numerice (ca `hstack` din NLP).
Ținta `y` = `Survived`. Afișează `X.shape`.

### Rezolvare 9

In [ ]:
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler

y = df["Survived"]

numeric_cols = ["Age", "Fare", "SibSp", "Parch", "Name_words", "HasCabin", "Age_missing"]
scaler = StandardScaler()
X_num = scaler.fit_transform(df[numeric_cols])

X_cat = pd.get_dummies(df[["Sex", "Embarked", "Title"]], drop_first=False)

tfidf20 = TfidfVectorizer(stop_words="english", max_features=20)
X_text = tfidf20.fit_transform(df["Name"])

X = hstack([
    csr_matrix(X_num),
    csr_matrix(X_cat.values.astype(float)),
    X_text,
])

print("Shape X:", X.shape)
print("Shape y:", y.shape)
print("Feature-uri numerice:", len(numeric_cols))
print("Feature-uri One-Hot:", X_cat.shape[1])
print("Feature-uri TF-IDF:", X_text.shape[1])

---
## Întrebarea 10

Antrenează un `LogisticRegression` (sklearn) pe `X`, `y`:
1. `train_test_split` (test_size=0.2, stratify=y, random_state=42);
2. raportează accuracy, classification_report și ROC-AUC;
3. afișează pentru 5 pasageri din validare: numele, clasa reală și **`P_survive`** (`predict_proba`).

Opțional: cross-validation 5-fold pe tot setul.

*Hint:* la rețete preziceam `difficulty`; aici prezicem probabilitatea de a fi supraviețuit.

### Rezolvare 10

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_val)
y_proba = clf.predict_proba(X_val)[:, 1]

print("Accuracy:", round(accuracy_score(y_val, y_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_val, y_proba), 3))
print("\nClassification report:")
print(classification_report(y_val, y_pred, target_names=["Died", "Survived"]))

preview = pd.DataFrame({
    "Name": df.loc[idx_val, "Name"].values,
    "Survived_real": y_val.values,
    "Survived_pred": y_pred,
    "P_survive": np.round(y_proba, 3),
})
preview.head()

In [ ]:
cv_acc = cross_val_score(clf, X, y, cv=5, scoring="accuracy")
cv_auc = cross_val_score(clf, X, y, cv=5, scoring="roc_auc")

print("CV accuracy:", np.round(cv_acc, 3), "| media:", round(cv_acc.mean(), 3))
print("CV ROC-AUC:", np.round(cv_auc, 3), "| media:", round(cv_auc.mean(), 3))

---
## Hartă rapidă: NLP rețete ↔ Titanic

| Concept în `02.nlp_recipes.ipynb` | Echivalent pe Titanic |
|----------------------------------|------------------------|
| `get_dummies` pe `mealType` | `get_dummies` pe `Sex`, `Embarked`, `Title` |
| `LabelEncoder` / `OrdinalEncoder` | `Sex`, `Pclass` |
| Frequency encoding | `Embarked_freq` |
| `name_len`, `name_words` | aceleași pe `Name` / `Ticket` |
| `CountVectorizer` / `TfidfVectorizer` | pe `Name` |
| `hstack` + LogisticRegression | predicție `Survived` + `predict_proba` |
| liste → string | aici: regex pe `Name`, `Deck` din `Cabin` |